# 08 — Residual Stream: The Model's Communication Highway

The **residual stream** is the central concept for understanding transformers. Every attention and MLP layer *reads from* and *writes to* this shared vector. The final prediction is determined by the sum of all contributions.

```
residual = embedding + Σ(attention_outputs) + Σ(mlp_outputs)
```

This notebook analyzes how information accumulates through the stream.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from utils.model_loading import load_tlens_model
from utils.visualization import apply_theme, ACCENT_BLUE, ACCENT_ORANGE, ACCENT_GREEN, ACCENT_RED, ACCENT_PURPLE, ACCENT_TEAL, TEXT_COLOR, DARK_BG, DARK_SURFACE, DARK_GRID, PALETTE, _style_box
apply_theme()

MODEL_SIZE = "0.5b"
model = load_tlens_model(MODEL_SIZE)
cfg = model.cfg

prompt = "The capital of France is"
logits, cache = model.run_with_cache(prompt)
tokens = model.to_str_tokens(prompt)
last_pos = len(tokens) - 1  # focus on last token position
print(f"Tokens: {tokens}")

## Residual Stream Norm Growth

The residual stream grows in magnitude as each layer adds its contribution.

In [ ]:
# Collect residual stream norms, attention output norms, and MLP output norms
resid_norms = []
attn_norms = []
mlp_norms = []

# Embedding norm
embed_resid = cache["hook_embed"][0, last_pos]
resid_norms.append(embed_resid.float().norm().item())

for i in range(cfg.n_layers):
    # Residual after this layer
    resid = cache[f"blocks.{i}.hook_resid_post"][0, last_pos]
    resid_norms.append(resid.float().norm().item())
    
    # Attention output (what attention adds to the stream)
    attn_out = cache[f"blocks.{i}.attn.hook_result"][0, last_pos]
    attn_norms.append(attn_out.float().norm().item())
    
    # MLP output
    mlp_out = cache[f"blocks.{i}.hook_mlp_out"][0, last_pos]
    mlp_norms.append(mlp_out.float().norm().item())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Residual stream norm growth
ax1.plot(range(len(resid_norms)), resid_norms, marker="o", markersize=4,
         color=ACCENT_TEAL, linewidth=2)
ax1.set_xlabel("Layer (0 = after embedding)")
ax1.set_ylabel("L2 Norm")
ax1.set_title("Residual Stream Norm Growth (last token)")
ax1.grid(alpha=0.2)

# Attention vs MLP contribution norms
x = np.arange(cfg.n_layers)
width = 0.35
ax2.bar(x - width/2, attn_norms, width, label="Attention", color=ACCENT_ORANGE, alpha=0.8)
ax2.bar(x + width/2, mlp_norms, width, label="MLP", color=ACCENT_GREEN, alpha=0.8)
ax2.set_xlabel("Layer")
ax2.set_ylabel("Output L2 Norm")
ax2.set_title("Attention vs MLP Contribution per Layer")
ax2.legend()
ax2.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

print(f"Residual stream grows from {resid_norms[0]:.1f} (embedding) to {resid_norms[-1]:.1f} (final layer)")
print(f"Mean attention contribution: {np.mean(attn_norms):.2f}")
print(f"Mean MLP contribution: {np.mean(mlp_norms):.2f}")

## Cosine Similarity Between Layers

How much does the residual stream change from one layer to the next?

In [ ]:
# Collect residual streams at each layer for last token
residuals = [cache["hook_embed"][0, last_pos].float().cpu()]
for i in range(cfg.n_layers):
    residuals.append(cache[f"blocks.{i}.hook_resid_post"][0, last_pos].float().cpu())

# Compute pairwise cosine similarity
n = len(residuals)
cos_sim = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        cos_sim[i, j] = torch.nn.functional.cosine_similarity(
            residuals[i].unsqueeze(0), residuals[j].unsqueeze(0)
        ).item()

fig, ax = plt.subplots(figsize=(10, 8))
labels = ["Embed"] + [f"L{i}" for i in range(cfg.n_layers)]
sns.heatmap(cos_sim, ax=ax, cmap="coolwarm", vmin=-1, vmax=1, square=True,
            xticklabels=labels, yticklabels=labels, annot=False)
ax.set_title("Cosine Similarity of Residual Stream Across Layers (last token)")
plt.tight_layout()
plt.show()

print("Adjacent layers are very similar (small updates).")
print("Early vs late layers are increasingly dissimilar — the representation transforms gradually.")

## PCA Trajectory: The Path Through Representation Space

By projecting the residual stream at each layer to 2D, we can visualize the "path" the representation takes from embedding to final prediction.

In [ ]:
# PCA of residual stream trajectory
resid_matrix = torch.stack(residuals).numpy()  # [n_layers+1, d_model]
pca = PCA(n_components=2)
projected = pca.fit_transform(resid_matrix)

fig, ax = plt.subplots(figsize=(10, 8))

# Plot trajectory as connected path
ax.plot(projected[:, 0], projected[:, 1], color=ACCENT_TEAL, linewidth=1, alpha=0.5)

# Color points by layer (gradient from blue to red)
cmap = plt.cm.coolwarm
for i in range(len(projected)):
    color = cmap(i / (len(projected) - 1))
    label = "Embed" if i == 0 else f"L{i-1}"
    ax.scatter(projected[i, 0], projected[i, 1], c=[color], s=60, zorder=5, edgecolors="white", linewidth=0.5)
    ax.annotate(label, (projected[i, 0], projected[i, 1]), fontsize=7,
                xytext=(5, 5), textcoords="offset points")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("Residual Stream Trajectory Through PCA Space (last token)")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("The residual stream traces a path through high-dimensional space.")
print("Blue = early layers, Red = late layers.")

## Decomposing the Final Residual

The final residual stream is exactly the sum of the embedding + all attention outputs + all MLP outputs. Let's verify this and show each component's relative contribution.

In [ ]:
# Verify the decomposition: final = embed + Σ attn + Σ mlp
embed = cache["hook_embed"][0, last_pos].float()
attn_sum = sum(cache[f"blocks.{i}.attn.hook_result"][0, last_pos].float() for i in range(cfg.n_layers))
mlp_sum = sum(cache[f"blocks.{i}.hook_mlp_out"][0, last_pos].float() for i in range(cfg.n_layers))

reconstructed = embed + attn_sum + mlp_sum
actual_final = cache[f"blocks.{cfg.n_layers - 1}.hook_resid_post"][0, last_pos].float()

error = (reconstructed - actual_final).norm().item()
print(f"Reconstruction error: {error:.6f} (should be ~0)")
print(f"  → The residual stream is exactly the sum of all component outputs.\n")

# Relative contributions (by norm)
embed_norm = embed.norm().item()
attn_total_norm = attn_sum.norm().item()
mlp_total_norm = mlp_sum.norm().item()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(["Embedding", "Σ Attention", "Σ MLP"],
              [embed_norm, attn_total_norm, mlp_total_norm],
              color=[ACCENT_BLUE, ACCENT_ORANGE, ACCENT_GREEN])
ax.set_ylabel("L2 Norm")
ax.set_title("Contribution to Final Residual Stream (by norm)")
for bar, val in zip(bars, [embed_norm, attn_total_norm, mlp_total_norm]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}", ha="center", fontsize=11)
plt.tight_layout()
plt.show()

## Summary

The residual stream view reveals:
1. **Additive composition**: The output is literally the sum of all component contributions
2. **Gradual transformation**: Adjacent layers produce similar representations; early vs late layers diverge
3. **Norm growth**: The stream accumulates signal with each layer
4. **MLP dominance**: MLP layers typically contribute more to the residual update than attention
5. **Smooth trajectory**: The representation traces a smooth path through high-dimensional space

This is why transformers are interpretable at the component level — every piece contributes additively to the same shared workspace.